In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from eval import load, passat
import pandas as pd

FIGURE_DIR = Path("../outputs/figures")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 14,
    "axes.labelsize": 14,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 14,
    "figure.dpi": 120,
    "savefig.dpi": 300,
})


In [ ]:
model_name = "allenai/OLMo-2-1124-7B-Instruct"
MODEL_LABEL = "OLMo 2 7B Instruct"
dataset_size = 164
numvariants = 51
max_k = 50

df_passk = load.load_humaneval(
    model_name=model_name,
    dataset_size=dataset_size,
    numvariants=numvariants,
    variant_type="temperature",
)

df_retok = load.load_humaneval(
    model_name=model_name,
    dataset_size=dataset_size,
    numvariants=numvariants,
    variant_type="retok",
)


In [ ]:
x_passk, y_passk, yerr_passk = passat.pass_curve_points(df_passk, max_k=max_k)

retok_ps = sorted(float(p) for p in df_retok["p"].dropna().unique() if float(p) > 0)
colors = plt.cm.plasma(np.linspace(0.15, 0.9, len(retok_ps)))
bins = np.linspace(0, 1, 25)

fig, axes = plt.subplots(1, 2, figsize=(10.5, 5.0), constrained_layout=True)
fig.set_constrained_layout_pads(w_pad=0.04, h_pad=0.08, hspace=0.12, wspace=0.08)
ax_curve, ax_dist = axes

ax_curve.fill_between(
    x_passk,
    y_passk - yerr_passk,
    y_passk + yerr_passk,
    color="black",
    alpha=0.08,
)
ax_curve.plot(x_passk, y_passk, color="black", linewidth=2.5, label="pass@k")

p_success_passk = df_passk.groupby("task_id").passed.mean().values
p_failure_passk = 1 - p_success_passk
counts, hist_bins = np.histogram(p_failure_passk, bins=bins)
counts = counts / counts.sum()
step_counts = np.append(counts, counts[-1])
ax_dist.step(hist_bins, step_counts, where="post", color="black", linewidth=2.5, label="pass@k")

ax_dist.axvline(np.mean(p_failure_passk), color="black", linestyle="--", linewidth=1)

for color, p_value in zip(colors, retok_ps):
    df_p = pd.concat([
        df_retok[df_retok["p"] == p_value].copy(),
        df_retok[df_retok["p"] == 0].copy(),
    ]).reset_index(drop=True)
    x_retok, y_retok, yerr_retok = passat.pass_curve_points(df_p, max_k=max_k)

    ax_curve.fill_between(
        x_retok,
        y_retok - yerr_retok,
        y_retok + yerr_retok,
        color=color,
        alpha=0.12,
    )
    ax_curve.plot(
        x_retok,
        y_retok,
        color=color,
        linewidth=2,
        label=f"pass@retok_{p_value:g}",
    )

    p_success_retok = df_p.groupby("task_id").passed.mean().values
    p_failure_retok = 1 - p_success_retok
    counts, hist_bins = np.histogram(p_failure_retok, bins=bins)
    counts = counts / counts.sum()
    step_counts = np.append(counts, counts[-1])
    ax_dist.step(
        hist_bins,
        step_counts,
        where="post",
        color=color,
        linewidth=2,
        label=f"pass@retok_{p_value:g}",
    )
    ax_dist.axvline(np.mean(p_failure_retok), color=color, linestyle="--", linewidth=1)

ax_curve.set_box_aspect(1)
# ax_curve.set_title("HumanEval")
ax_curve.set_xlabel("k")
ax_curve.set_ylabel("Pass Rate")
ax_curve.set_xlim(1, max_k)
ax_curve.grid(True, alpha=0.3)
ax_curve.legend(loc="lower right", frameon=False)

ax_dist.set_box_aspect(1)
# ax_dist.set_title("HumanEval")
ax_dist.set_xlabel(r"$P_{fail}$")
ax_dist.set_ylabel("Fraction of Tasks")
ax_dist.set_xlim(0, 1)
ax_dist.set_xticks(np.linspace(0, 1, 6))
ax_dist.grid(True, alpha=0.3)
# ax_dist.legend(loc="upper left", frameon=False)

# fig.suptitle(MODEL_LABEL, y=0.995)
fig.savefig(FIGURE_DIR / "passat_retok_per_p_humaneval.svg", bbox_inches="tight")
fig.savefig(FIGURE_DIR / "passat_retok_per_p_humaneval.png", bbox_inches="tight")
plt.show()
